In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"

excel_files = sorted(
    path
    for path in RAW_DIR.glob("*.xlsx")
    if not path.name.startswith("~$")
)

if not excel_files:
    raise FileNotFoundError(f"No .xlsx files found in {RAW_DIR}")

for index, path in enumerate(excel_files):
    print(f"{index}: {path.name}")

# Change this index if the intended workbook isn't first.
DATA_PATH = excel_files[0]
print("\nSelected:", DATA_PATH.name)

0: July 2026 Complete Monthly Ridership (with adjustments and estimates)_260901.xlsx

Selected: July 2026 Complete Monthly Ridership (with adjustments and estimates)_260901.xlsx


In [3]:
NTD_ID = "00001"  # King County
MODE = "MB"       # Motorbus
TOS = "DO"        # Directly operated

upt = pd.read_excel(
    DATA_PATH,
    sheet_name="UPT",
    dtype={"NTD ID": "string"},
)

seattle = upt.loc[
    upt["NTD ID"].eq(NTD_ID)
    & upt["Mode"].eq(MODE)
    & upt["TOS"].eq(TOS)
].copy()

if len(seattle) != 1:
    raise ValueError(
        f"Expected one matching Seattle row, found {len(seattle)}."
    )

display(seattle[["NTD ID", "Agency", "Mode", "TOS"]])

,NTD ID,Agency,Mode,TOS
4,00001,King County,MB,DO


In [5]:
# Identify month columns such as "1/2002" and "7/2026".
month_columns = [
    column
    for column in upt.columns
    if pd.Series([str(column)]).str.fullmatch(
        r"\d{1,2}/\d{4}"
    ).iloc[0]
]

monthly = seattle.melt(
    value_vars=month_columns,
    var_name="date",
    value_name="ridership",
)

monthly["date"] = pd.to_datetime(
    monthly["date"],
    format="%m/%Y",
)

monthly["ridership"] = pd.to_numeric(
    monthly["ridership"],
    errors="raise",
)

monthly = (
    monthly
    .set_index("date")
    .sort_index()
    .asfreq("MS")  # One observation per calendar month.
)

print(f"Number of months: {len(monthly)}")
print(f"Missing ridership values: {monthly['ridership'].isna().sum()}")

display(monthly.head())
display(monthly.tail())

Number of months: 295
Missing ridership values: 0


,ridership
date,
2002-01-01,6045861.0
2002-02-01,5406135.0
2002-03-01,5999230.0
2002-04-01,6058398.0
2002-05-01,6134503.0


,ridership
date,
2026-03-01,6646228.0
2026-04-01,6738016.0
2026-05-01,6932761.0
2026-06-01,6611961.0
2026-07-01,6298063.0


In [6]:
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / (
    f"ntd_{NTD_ID}_{MODE}_{TOS}_monthly.csv"
)

monthly.to_csv(output_path, index_label="date")

print(f"Saved {len(monthly)} months to {output_path}")

Saved 295 months to /Users/alliesewell/transit_dfc_mlpl/data/processed/ntd_00001_MB_DO_monthly.csv
